# Bước 4 — Chung kết trên Lightning AI
Giữ **Studio T4 đã chạy Bước 3**, upload notebook này và **step4_bundle.zip** vào
cùng thư mục với notebook Bước 3. Không cần upload lại trọng số T05.
Nếu Studio mới, upload thêm **step3_artifacts.zip** đã tải từ Bước 3 vào cùng thư mục.

Cần **6 lượt train mới**, 10 epoch/lượt: F00=T00/I00 và F01=T05/I09, mỗi nhóm seed 0/1/2.
Dự trù **1.5–2 giờ**, ước lượng từ các lượt trước, chưa phải thời gian đo Lightning.
F01 khớp T riêng trên val 288 trước test; F01 calibrated là kết quả chính.
Không lấy nhiệt độ I07 ở 224 áp cho 288.

Chạy sáu ô train/val trước, rồi mới chạy test. Test đúng một forward pass mỗi model/seed.
Chạy lại chỉ dùng cache test; nếu test dở không có cache hoàn chỉnh, notebook dừng.
Không đổi công thức/frozen_plan sau khi mở test.
Cuối phiên tải **step4_artifacts.zip**, **step4_best_checkpoints.zip** và notebook có output.


In [ ]:
import sys, subprocess, json, base64, hashlib, shutil, os
from pathlib import Path
if sys.version_info < (3,10):
    raise RuntimeError('Notebook yêu cầu Python >=3.10.')
# Keep the CUDA-enabled torch/torchvision supplied by the Studio.
try:
    import torch, torchvision
except ImportError as exc:
    raise RuntimeError('Thiếu torch/torchvision. Trong terminal Studio cài: '
        'pip install torch==2.11.0 torchvision==0.26.0 --index-url https://download.pytorch.org/whl/cu126 '
        'rồi restart kernel và chạy lại.') from exc
if not torch.cuda.is_available():
    raise RuntimeError('Bật machine T4 và chọn kernel có CUDA rồi chạy lại. '
        'Nếu đang dùng torch CPU, cài bộ CUDA trong hướng dẫn và restart kernel.')
print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__,
      '| torchvision:', torchvision.__version__, '| CUDA:', torch.version.cuda)
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'timm==1.0.30', 'fvcore==0.1.5.post20221221', 'matplotlib', 'pandas',
    'pillow', 'scikit-learn', 'openpyxl'], capture_output=True, text=True)
print(result.stdout, result.stderr)
result.check_returncode()
if 'timm' in sys.modules and sys.modules['timm'].__version__ != '1.0.30':
    raise RuntimeError('Đã cài timm 1.0.30; restart kernel rồi chạy lại từ đầu.')
NOTEBOOK_DIR = Path.cwd().resolve()
BASE_DIR = NOTEBOOK_DIR / 'deepweeds_step3_lightning'
PROJECT_DIR = BASE_DIR / 'project'
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print('Thư mục lưu dữ liệu/kết quả:', BASE_DIR)


In [ ]:
import zipfile
PATCH_SHA256 = '1909fcc0daa5663ba19fc0a7c349b6a878791c8dd503ed536d49bb30af38a2c4'
PREVIOUS_SHA256 = '9b143ede6f25011a4fc549811b895d07cae03ec2faeb22d6f57dbd67eac1b81c'
def stream_sha256(stream):
    digest=hashlib.sha256()
    for chunk in iter(lambda:stream.read(1<<20),b''):
        digest.update(chunk)
    return digest.hexdigest()
def checked_unpack(path,expected,*,patch=False):
    if not path.is_file():
        raise FileNotFoundError(f'Upload {path.name} vào {NOTEBOOK_DIR} rồi chạy lại.')
    with path.open('rb') as stream:
        if stream_sha256(stream)!=expected:
            raise ValueError(f'ZIP không khớp notebook: {path.name}')
    with zipfile.ZipFile(path) as bundle:
        for info in bundle.infolist():
            if info.is_dir():
                continue
            target=(PROJECT_DIR/info.filename).resolve()
            if not target.is_relative_to(PROJECT_DIR.resolve()):
                raise ValueError('Invalid ZIP path')
            if target.exists() and not (patch and info.filename.startswith(('code/','tests/'))):
                existing=target.read_bytes()
                supplied=bundle.read(info)
                matches=existing==supplied
                if patch and info.filename=='step4/frozen_plan.json':
                    # Windows CRLF and Lightning LF encode the same JSON protocol.
                    # Keep the existing bytes: val/test signatures hash this file.
                    matches=json.loads(existing)==json.loads(supplied)
                if patch and not matches:
                    raise FileExistsError(f'Existing frozen protocol differs: {target}')
                continue
            target.parent.mkdir(parents=True,exist_ok=True)
            with bundle.open(info) as supplied,target.open('wb') as destination:
                shutil.copyfileobj(supplied,destination)
if not (PROJECT_DIR/'step3/selection.json').is_file():
    checked_unpack(NOTEBOOK_DIR/'step3_artifacts.zip',PREVIOUS_SHA256)
checked_unpack(NOTEBOOK_DIR/'step4_bundle.zip',PATCH_SHA256,patch=True)
CODE_DIR=PROJECT_DIR/'code'
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0,str(CODE_DIR))
for name in ['dataset','model','losses','train','inference','benchmark','step0','step1','step2','step3','step4']:
    sys.modules.pop(name,None)
import step0,step4
# The bundle already contains the frozen plan. Preserve its exact bytes on resume
# because completed val/test records refer to this file's SHA-256.
plan=step4.plan_at(PROJECT_DIR)
print(json.dumps(plan,ensure_ascii=False,indent=2))
print('Runtime:',step4.runtime_check(PROJECT_DIR))


In [ ]:
command=[sys.executable,'-X','utf8','-c',
    "import torch,unittest; torch.set_num_threads(2); s=unittest.defaultTestLoader.discover('tests'); "
    "r=unittest.TextTestRunner(verbosity=1).run(s); raise SystemExit(not r.wasSuccessful())"]
result=subprocess.run(command,cwd=PROJECT_DIR,capture_output=True,text=True)
print(result.stdout,result.stderr); result.check_returncode()


In [ ]:
IMAGES_DIR,LABELS_DIR=step0.prepare_data(BASE_DIR)
print('Chỉ kiểm tra metadata test trong giai đoạn train. Chưa tạo dự đoán test.')


## F00 — seed 0
Train 10 epoch và khóa val/T/latency.


In [ ]:
summary=step4.run_training(PROJECT_DIR,IMAGES_DIR,LABELS_DIR,'F00',0)
val=step4.prepare_val(PROJECT_DIR,IMAGES_DIR,LABELS_DIR,'F00',0)
print(json.dumps(dict(training=summary,val=val),ensure_ascii=False,indent=2))


## F01 — seed 0
Train 10 epoch và khóa val/T/latency.


In [ ]:
summary=step4.run_training(PROJECT_DIR,IMAGES_DIR,LABELS_DIR,'F01',0)
val=step4.prepare_val(PROJECT_DIR,IMAGES_DIR,LABELS_DIR,'F01',0)
print(json.dumps(dict(training=summary,val=val),ensure_ascii=False,indent=2))


## F00 — seed 1
Train 10 epoch và khóa val/T/latency.


In [ ]:
summary=step4.run_training(PROJECT_DIR,IMAGES_DIR,LABELS_DIR,'F00',1)
val=step4.prepare_val(PROJECT_DIR,IMAGES_DIR,LABELS_DIR,'F00',1)
print(json.dumps(dict(training=summary,val=val),ensure_ascii=False,indent=2))


## F01 — seed 1
Train 10 epoch và khóa val/T/latency.


In [ ]:
summary=step4.run_training(PROJECT_DIR,IMAGES_DIR,LABELS_DIR,'F01',1)
val=step4.prepare_val(PROJECT_DIR,IMAGES_DIR,LABELS_DIR,'F01',1)
print(json.dumps(dict(training=summary,val=val),ensure_ascii=False,indent=2))


## F00 — seed 2
Train 10 epoch và khóa val/T/latency.


In [ ]:
summary=step4.run_training(PROJECT_DIR,IMAGES_DIR,LABELS_DIR,'F00',2)
val=step4.prepare_val(PROJECT_DIR,IMAGES_DIR,LABELS_DIR,'F00',2)
print(json.dumps(dict(training=summary,val=val),ensure_ascii=False,indent=2))


## F01 — seed 2
Train 10 epoch và khóa val/T/latency.


In [ ]:
summary=step4.run_training(PROJECT_DIR,IMAGES_DIR,LABELS_DIR,'F01',2)
val=step4.prepare_val(PROJECT_DIR,IMAGES_DIR,LABELS_DIR,'F01',2)
print(json.dumps(dict(training=summary,val=val),ensure_ascii=False,indent=2))


## Test cuối cùng
Chỉ chạy sau khi sáu ô train/val đều hoàn thành. Không sửa cấu hình sau ô này.
Một lượt model mỗi nhóm/seed; bản calibrated/uncal lấy từ cùng logits đã lưu.


In [ ]:
for seed in step4.SEEDS:
    for group in step4.GROUPS:
        result=step4.test_once(PROJECT_DIR,IMAGES_DIR,LABELS_DIR,group,seed)
        print(group,seed,json.dumps(result['metrics'],ensure_ascii=False))


In [ ]:
summary=step4.summarize(PROJECT_DIR,LABELS_DIR)
print(json.dumps(summary,ensure_ascii=False,indent=2))
print((PROJECT_DIR/'step4/eval_outputs/eval_output.md').read_text(encoding='utf-8'))
from IPython.display import display,Image,FileLink
for group in step4.GROUPS:
    display(Image(filename=str(PROJECT_DIR/f'step4/{group}_confusion.png')))


In [ ]:
artifacts=step4.export(PROJECT_DIR,BASE_DIR/'step4_artifacts.zip')
weights=step4.export(PROJECT_DIR,BASE_DIR/'step4_best_checkpoints.zip',checkpoints=True)
for archive in (artifacts,weights):
    print('Download:',archive)
    display(FileLink(str(archive.relative_to(NOTEBOOK_DIR))))
print('Có thể tải bằng chuột phải file → Download trong Files. Lưu thêm notebook có output.')


Sau khi tải hai ZIP, gửi về dự án để nhập/đối chiếu. Bước 5 làm trên CPU:
hoàn thiện Final/PerClass/Summary, mean ± std, report.md, ma trận nhầm lẫn, ảnh lỗi,
README tái lập và push Git. Không cần train thêm để tổng hợp Bước 5.
